<a href="https://colab.research.google.com/github/Franco623/MiniProjetoFrancoUmilioAnalisedeDadosT7/blob/main/analise_varejo_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Primeira parte do trabalho:
​Carregar​ ​a ​​base​​Varejo.csv ​​com​ ​pandas​ ​e ​​mostrar:​​número​ ​de​ ​registros,​​colunas
​​e​ tipos​ ​de dados.​

In [ ]:
 #importando biblioteca pandas
import pandas as pd
import numpy as np
# Carregando base de dados.csv
# Com sep=';' a saida mostrava - Quantidade de colunas: 1
# Usando sep=',' para otimizar a importação do csv e mostrar o correto
# Quantidade de colunas 14

#df_varejo = pd.read_csv("Base_Varejo.csv", sep = ":")

df_varejo = pd.read_csv("/content/Base_Varejo.csv", sep = ",")

#  Mostra a quantidade de registros e colunas
print("\n--- DIMENSÕES DA BASE ---")
print(f"Quantidade de registros: {df_varejo.shape[0]}")
print(f"Quantidade de colunas: {df_varejo.shape[1]}")

# Mostra os nomes das colunas
print("\n--- Nomes das colunas ---")
print(df_varejo.columns.tolist())

# Mostra os tipos de dados
print("\n--- TIPOS DE DADOS ---")
print(df_varejo.dtypes)

# Mostra as primeiras linhas
print ("\n--- PRIMEIRAS LINHAS DA BASE ---")
print(df_varejo.head())



A saida acima detectou 4 colunas Unnamed (sem nome) Olhando o arquivo  Base_Varejo.csv, percebi que algumas linhas Unnamed (sem nome) são textos continuação da coluna anterior. Como uso LibreOffice em Debian 12, pode ser que baixando o arquivo csv da origine, uma malformação de configuração aconteceu ou a base é assim na origem.



Mundando os nomes Unnamed para Sem_Nome usando un for para percorrer as colunas é un if else para achar "Unnamed" e tracar para "Sem_Nome" o código a seguir exibe antes e depois


In [3]:
print("--- Antes ---")
print(df_varejo.columns.tolist())

novas_colunas = []
for coluna in df_varejo.columns:
    if "Unnamed" in str(coluna):
        novas_colunas.append("Sem_Nome")
    else:
        novas_colunas.append(coluna)

df_varejo.columns = novas_colunas

print("\n--- Depois ---")
print(df_varejo.columns.tolist())



--- Antes ---
['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']

--- Depois ---
['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME', 'Sem_Nome', 'Sem_Nome', 'Sem_Nome', 'Sem_Nome']


Segue um print da tabela com nomes corrigidos e soma os valores nulos existentes.


In [ ]:
# Reescrivendo os nomes completos dos produtos
print("\n--- CORRIGINDO OS NOMES DOS PRODUTOS ---")

colunas_produto = [
    "PR_NOME",
    "Sem_Nome",
]

# Juntar as partes dos nomes, ignorando campos vazios
df_varejo["PR_NOME"] = (
    df_varejo[colunas_produto]

    .fillna("") # Substitui valores nulos (NaN) por string vazia.

    .astype(str) # Converte todos os valores para texto.

    .agg(" ".join, axis=1) # Junta os valores de cada linha usando
    #um espaço entre eles.

    .str.replace(r"\s+", " ", regex=True) # Substitui múltiplos espaços
    #em branco por um único espaço.

    .str.strip() # Remove espaços em branco no início e no final da string.
)

# Excluir as colunas auxiliares após reconstruir os nomes
df_varejo = df_varejo.drop(
    columns=colunas_produto[1:]
)

print("Quantidade de colunas após a correção:",
      df_varejo.shape[1])


Foram analisadas todas as colunas . Não foram encontradas categorias vazias nas variáveis CL_GENERO, CL_SEG, PR_CAT e PR_NOME. Entretanto, foram identificadas colunas Sem_Nome, contendo grande quantidade de valores "null" ou em algumas linhas de trecho de texto das linhas anteriores. Estando com valores nulos podem ser deletadas não antes de reconstruir as colunas PR_NOME para uma melhor leitura da base de dados



Reconstruidas as colunas  os valores null são deletados e a seguir
as colunas passam de 14 para 10  sem valores nulos dectettados.


In [ ]:
# exibe a base de dados corrigida
print(df_varejo)

# soma valores nulos e exibe

valores_nulos = df_varejo.isnull().sum()
print(valores_nulos)

Procurando valores nulos

In [ ]:

valores_nulos = df_varejo.isnull().sum()
print(valores_nulos)

Procurando categorias vazias


In [ ]:
print(df_varejo["PR_CAT"].value_counts(dropna=False))


Substituição dos "#N/D" por "Sem_Categoria"

In [ ]:
print(df_varejo["PR_CAT"].unique())
print("-----"*20)
nova_categoria = []
for valor in df_varejo['PR_CAT']:
    if valor == "#N/D":
        nova_categoria.append("Sem_Categoria")
    else:
        nova_categoria.append(valor)

df_varejo["PR_CAT"] = nova_categoria

print(df_varejo["PR_CAT"].unique())

Registros Duplicados

In [ ]:
print(" REGISTROS DUPLICADOS")
duplicados = df_varejo.duplicated().sum()
print(f"Quantidade de registros duplicados: {duplicados}")

duplicadas = df_varejo[df_varejo.duplicated(keep=False)]
print(duplicadas)

Removendo Duplicadas con .drop

In [ ]:
print(" Removendo duplicatas...")

quantidade_duplicados = df_varejo.duplicated().sum()

df_varejo = df_varejo.drop_duplicates()

print(f"Duplicatas removidas: {quantidade_duplicados}")

Tratando possiveis inconsitências datas invalidas

In [ ]:

print(df_varejo['DATA'].head())

#As datas são escritas no formato "01/01/01"

# Depois do passo a seguir as datas serão transformadas em "datatime" padrão
df_varejo['DATA'] = pd.to_datetime(
    df_varejo['DATA'],
    format='%d/%m/%y',
    errors='coerce'
)
print("*"*40)
#datas_invalidas = df_varejo['DATA'].isna().sum()

#print(f"Datas inválidas: {datas_invalidas}")

print(df_varejo['DATA'].head())

ESTATÍSTICAS DESCRITIVAS


In [ ]:
coluna = "CL_FHL"

print(" ESTATÍSTICAS DESCRITIVAS - NÚMERO DE FILHOS")


print(f"Contagem: {df_varejo[coluna].count()}")
print(f"Média: {df_varejo[coluna].mean():.2f}")
print(f"Mediana: {df_varejo[coluna].median()}")
print(f"Moda: {df_varejo[coluna].mode().iloc[0]}")
print(f"Desvio padrão: {df_varejo[coluna].std():.2f}")
print(f"Mínimo: {df_varejo[coluna].min()}")
print(f"Máximo: {df_varejo[coluna].max()}")

# Quartis 25%,50%,75%

print("\nQuartis:")
print(f"Q1 (25%): {df_varejo[coluna].quantile(0.25)}")
print(f"Q2 (50%): {df_varejo[coluna].quantile(0.50)}")
print(f"Q3 (75%): {df_varejo[coluna].quantile(0.75)}")

print("-" * 30)
print(df_varejo[coluna].value_counts().sort_index())


Foram calculadas estatísticas descritivas para a variável CL_FHL que representa o número de filhos dos clientes.
A análise incluiu medidas de tendência central (média, mediana e moda) e desvio padrão, valores mínimo e máximo, contagem de registros válidos e quartis (25%, 50% e 75%).
A variável CL_FHL apresenta forte concentração em clientes sem filhos. Mais de 50% dos registros possuem valor 0, o que explica a moda e a mediana iguais a zero. Outro ponto: 75% dos clientes possuem até 2 filhos, conforme indicado pelo terceiro quartil (Q3 = 2).

Agrupamento 1 - CL_GENERO x Quantidade de clientes ---
Agrupamento 2 - CL_GENERO x  Média de Filhos

In [ ]:
# AGRUPAMENTO 1
clientes_genero = df_varejo.groupby("CL_GENERO").size()

print("Quantidade de clientes por gênero:")
print(clientes_genero)

# AGRUPAMENTO 2
media_filhos_genero = df_varejo.groupby("CL_GENERO")["CL_FHL"].mean()

print("\nMédia de filhos por gênero:")
print(media_filhos_genero)

Agrupamento 1 - Foi realizado um agrupamento da base pela variável CL_GENERO, contabilizando a quantidade de clientes em cada categoria.
Essa análise permitiu identificar a distribuição dos clientes por gênero e verificar qual grupo apresenta maior representatividade na base.


Agrupamento 2 -  Um segundo agrupamento por gênero utilizando a variável CL_FHL. Foi calculada a média do número de filhos para cada grupo, permitindo comparar características demográficas entre os clientes.

Salvar Base Limpa

In [ ]:
df_varejo.to_csv("dados_limpos.csv", index=False)

print("\nArquivo 'dados_limpos.csv' gerado com sucesso.")

CONCLUSÕES

• Foram tratados valores nulos, removidas duplicatas e ajustados os tipos de dados da base.
• A coluna DATA foi convertida para datetime, permitindo análises temporais adequadas.
• A variável CL_FHL foi analisada por meio de estatísticas descritivas, revelando sua distribuição na base.
• Os agrupamentos realizados permitiram identificar: 1) a distribuição dos clientes por gênero e verificar qual grupo apresenta maior representatividade na base; 2)  Foi calculada a média do número de filhos para cada grupo permitindo comparar características demográficas entre os clientes.
• A qualidade dos dados melhorou após as etapas de limpeza.
• Ainda podem existir inconsistências cadastrais e possíveis outliers que exigem análises futura.
